# 반응-확산 시스템 (Reaction-Diffusion)

이 노트북은 KooLab을 사용한 반응-확산 시스템 시뮬레이션을 다룹니다.

## 목차
1. Gray-Scott 모델
2. Brusselator 시스템
3. 패턴 형성 시각화
4. 파라미터 스터디
5. 애니메이션

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'build'))

import _core as koo
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

koo.Logger.initialize("ReactionDiffusion")
koo.Logger.set_level(koo.LogLevel.INFO)
print(f"KooLab Version: {koo.version()}")

## 1. Gray-Scott 모델

Gray-Scott 모델은 다음과 같은 반응-확산 방정식을 따릅니다:

$$
\begin{aligned}
\frac{\partial u}{\partial t} &= D_u \nabla^2 u - uv^2 + F(1-u) \\
\frac{\partial v}{\partial t} &= D_v \nabla^2 v + uv^2 - (F+k)v
\end{aligned}
$$

여기서:
- $u$, $v$: 두 화학종의 농도
- $D_u$, $D_v$: 확산 계수
- $F$: feed rate
- $k$: kill rate

In [ ]:
class GrayScottSimulator:
    def __init__(self, size=128, Du=0.16, Dv=0.08, F=0.060, k=0.062, dt=1.0):
        self.size = size
        self.Du = Du
        self.Dv = Dv
        self.F = F
        self.k = k
        self.dt = dt
        
        # 초기 조건
        self.u = np.ones((size, size))
        self.v = np.zeros((size, size))
        
        # 중앙에 작은 섭동 추가
        center = size // 2
        r = 10
        for i in range(center-r, center+r):
            for j in range(center-r, center+r):
                if (i-center)**2 + (j-center)**2 < r**2:
                    self.u[i, j] = 0.50
                    self.v[i, j] = 0.25
    
    def laplacian(self, field):
        """5점 스텐실 라플라시안"""
        lapl = (
            np.roll(field, 1, axis=0) + 
            np.roll(field, -1, axis=0) + 
            np.roll(field, 1, axis=1) + 
            np.roll(field, -1, axis=1) - 
            4 * field
        )
        return lapl
    
    def step(self):
        """한 시간 스텝 진행"""
        uvv = self.u * self.v * self.v
        
        du = self.Du * self.laplacian(self.u) - uvv + self.F * (1 - self.u)
        dv = self.Dv * self.laplacian(self.v) + uvv - (self.F + self.k) * self.v
        
        self.u += du * self.dt
        self.v += dv * self.dt
    
    def run(self, steps):
        """여러 스텝 실행"""
        for _ in range(steps):
            self.step()

print("GrayScottSimulator class defined")

## 2. 시뮬레이션 실행

In [ ]:
# 시뮬레이터 생성
sim = GrayScottSimulator(size=128, Du=0.16, Dv=0.08, F=0.060, k=0.062)

koo.Logger.info("Starting Gray-Scott simulation")

# 여러 시간에서의 스냅샷 저장
snapshots = [sim.v.copy()]
times = [0]

for i in range(1, 11):
    sim.run(1000)
    snapshots.append(sim.v.copy())
    times.append(i * 1000)
    if i % 2 == 0:
        koo.Logger.info(f"Completed {i * 1000} steps")

print(f"Simulation complete. Saved {len(snapshots)} snapshots.")

## 3. 패턴 형성 시각화

In [ ]:
# 4개의 타임 스냅샷 플롯
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
indices = [0, 3, 6, 9]

for ax, idx in zip(axes.flat, indices):
    im = ax.imshow(snapshots[idx], cmap='RdYlBu_r', vmin=0, vmax=0.5)
    ax.set_title(f'Step {times[idx]:,}', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Gray-Scott Pattern Formation', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

koo.Logger.info("Pattern visualization complete")

## 4. 파라미터 스터디: Feed Rate (F) 변화

In [ ]:
# 다양한 F 값에 대한 패턴
F_values = [0.030, 0.045, 0.060, 0.075]
k_fixed = 0.062
steps = 5000

fig, axes = plt.subplots(2, 2, figsize=(14, 14))

koo.Logger.info("Starting parameter study for F values")

for ax, F in zip(axes.flat, F_values):
    sim = GrayScottSimulator(size=128, Du=0.16, Dv=0.08, F=F, k=k_fixed)
    sim.run(steps)
    
    im = ax.imshow(sim.v, cmap='viridis', vmin=0, vmax=0.5)
    ax.set_title(f'F = {F:.3f}', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
    
    koo.Logger.debug(f"Completed simulation for F={F}")

plt.suptitle(f'Parameter Study: Feed Rate (F) at k={k_fixed}', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

koo.Logger.info("Parameter study complete")

## 5. 파라미터 공간 맵

다양한 (F, k) 조합에서 형성되는 패턴 유형을 탐색합니다.

In [ ]:
# 파라미터 그리드
F_grid = [0.020, 0.040, 0.060, 0.080]
k_grid = [0.045, 0.055, 0.065, 0.075]
steps = 4000

fig, axes = plt.subplots(4, 4, figsize=(16, 16))

koo.Logger.info("Starting parameter space exploration")

for i, F in enumerate(F_grid):
    for j, k in enumerate(k_grid):
        ax = axes[i, j]
        sim = GrayScottSimulator(size=64, Du=0.16, Dv=0.08, F=F, k=k, dt=1.0)
        sim.run(steps)
        
        im = ax.imshow(sim.v, cmap='hot', vmin=0, vmax=0.5)
        ax.set_title(f'F={F:.3f}, k={k:.3f}', fontsize=9)
        ax.axis('off')

plt.suptitle('Gray-Scott Parameter Space (F vs k)', 
             fontsize=18, fontweight='bold', y=0.998)
plt.tight_layout()
plt.show()

koo.Logger.info("Parameter space exploration complete")

## 6. Brusselator 시스템

Brusselator는 또 다른 유명한 반응-확산 시스템입니다:

$$
\begin{aligned}
\frac{\partial u}{\partial t} &= D_u \nabla^2 u + a - (b+1)u + u^2v \\
\frac{\partial v}{\partial t} &= D_v \nabla^2 v + bu - u^2v
\end{aligned}
$$

In [ ]:
class BrusselatorSimulator:
    def __init__(self, size=128, Du=0.5, Dv=1.0, a=1.0, b=3.0, dt=0.01):
        self.size = size
        self.Du = Du
        self.Dv = Dv
        self.a = a
        self.b = b
        self.dt = dt
        
        # 초기 조건: 균일 상태 + 랜덤 섭동
        self.u = self.a + 0.1 * np.random.randn(size, size)
        self.v = self.b / self.a + 0.1 * np.random.randn(size, size)
    
    def laplacian(self, field):
        lapl = (
            np.roll(field, 1, axis=0) + 
            np.roll(field, -1, axis=0) + 
            np.roll(field, 1, axis=1) + 
            np.roll(field, -1, axis=1) - 
            4 * field
        )
        return lapl
    
    def step(self):
        uuv = self.u * self.u * self.v
        
        du = self.Du * self.laplacian(self.u) + self.a - (self.b + 1) * self.u + uuv
        dv = self.Dv * self.laplacian(self.v) + self.b * self.u - uuv
        
        self.u += du * self.dt
        self.v += dv * self.dt
    
    def run(self, steps):
        for _ in range(steps):
            self.step()

print("BrusselatorSimulator class defined")

In [ ]:
# Brusselator 시뮬레이션
koo.Logger.info("Starting Brusselator simulation")

bruss = BrusselatorSimulator(size=128, Du=0.5, Dv=1.0, a=1.0, b=3.0)
bruss.run(5000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

im1 = ax1.imshow(bruss.u, cmap='RdBu_r')
ax1.set_title('Species U', fontsize=14, fontweight='bold')
ax1.axis('off')
plt.colorbar(im1, ax=ax1, fraction=0.046)

im2 = ax2.imshow(bruss.v, cmap='RdBu_r')
ax2.set_title('Species V', fontsize=14, fontweight='bold')
ax2.axis('off')
plt.colorbar(im2, ax=ax2, fraction=0.046)

plt.suptitle('Brusselator Pattern Formation', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

koo.Logger.info("Brusselator visualization complete")

## 7. 통계 분석

In [ ]:
# Gray-Scott 시스템에 대한 통계
sim = GrayScottSimulator(size=128)

mean_u = []
mean_v = []
std_v = []
time_points = []

for i in range(0, 10001, 100):
    if i > 0:
        sim.run(100)
    mean_u.append(np.mean(sim.u))
    mean_v.append(np.mean(sim.v))
    std_v.append(np.std(sim.v))
    time_points.append(i)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(time_points, mean_u, label='Mean(u)', linewidth=2)
ax1.plot(time_points, mean_v, label='Mean(v)', linewidth=2)
ax1.set_xlabel('Time Step')
ax1.set_ylabel('Mean Concentration')
ax1.set_title('Mean Concentrations Over Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(time_points, std_v, color='red', linewidth=2)
ax2.set_xlabel('Time Step')
ax2.set_ylabel('Std(v)')
ax2.set_title('Pattern Variability (Standard Deviation of v)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final mean(u) = {mean_u[-1]:.4f}, mean(v) = {mean_v[-1]:.4f}")
print(f"Final std(v) = {std_v[-1]:.4f}")

## 8. 요약

이 노트북에서 배운 내용:
- ✅ Gray-Scott 반응-확산 시스템 구현
- ✅ Brusselator 시스템 구현
- ✅ 패턴 형성 시각화
- ✅ 파라미터 스터디 수행
- ✅ 통계 분석

### 주요 발견:
- Feed rate (F)와 kill rate (k)의 조합이 패턴 유형을 결정
- 다양한 파라미터 조합으로 점, 줄무늬, 미로 등의 패턴 생성 가능
- 시간에 따라 패턴이 안정화되는 과정 관찰

### 다음 단계:
- `03_real_time_viz.ipynb`: 실시간 시각화
- `04_gpu_acceleration.ipynb`: GPU 가속

In [ ]:
koo.Logger.info("Reaction-Diffusion tutorial completed successfully!")